In [1]:
import pandas as pd
import numpy as np
import re

In [11]:
FILES = [
    r"Entso_e\DA_price\GUI_ENERGY_PRICES_202312312300-202412312300.csv",
    r"Entso_e\DA_price\GUI_ENERGY_PRICES_202412312300-202512312300.csv",
    r"Entso_e\DA_price\GUI_ENERGY_PRICES_202512312300-202612312300.csv",
    r"Entso_e\DA_price\GUI_ENERGY_PRICES_202309152200-202309162200.csv",
]

PRICE_COL_CANDIDATES = [
    "Day-ahead Price (EUR/MWh)",
    "Day-ahead Price (EUR/MWh )",
    "Day-ahead Price(EUR/MWh)",
]

MTU_COL = "MTU (CET/CEST)"
TZ_TARGET = "Europe/Budapest"
TZ = "Europe/Budapest"

In [14]:
def parse_dt_with_label(s: str) -> pd.Timestamp:
    """
    Input example: '31/03/2024 01:00:00 (CET)' or '(CEST)'
    Returns tz-aware timestamp in Europe/Budapest.
    """
    s = s.strip()
    m = re.match(r"^(.*?)(?:\s+\((CET|CEST)\))?$", s)
    if not m:
        raise ValueError(f"Unrecognized datetime format: {s}")
    dt_part = m.group(1).strip()
    label = m.group(2)  # 'CET' or 'CEST' or None

    dt = pd.to_datetime(dt_part, format="%d/%m/%Y %H:%M:%S", errors="raise")

    # If label missing, fall back to localize without infer (choose a safe policy)
    if label is None:
        # safest conservative fallback: treat as Europe/Budapest and shift forward if nonexistent
        return dt.tz_localize(TZ_TARGET, ambiguous="NaT", nonexistent="shift_forward")

    offset_hours = 1 if label == "CET" else 2  # CET=UTC+1, CEST=UTC+2
    # Convert local time with known offset -> UTC -> Europe/Budapest
    utc = (dt - pd.Timedelta(hours=offset_hours)).tz_localize("UTC")
    return utc.tz_convert(TZ_TARGET)

def parse_mtu_start_end(mtu: str):
    a, b = [x.strip() for x in mtu.split("-")]
    start = parse_dt_with_label(a)
    end   = parse_dt_with_label(b)
    return start, end

In [15]:
def pick_price_col(df: pd.DataFrame) -> str:
    for c in PRICE_COL_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"Could not find day-ahead price column. Available columns: {list(df.columns)}")

In [16]:
def load_one(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if MTU_COL not in df.columns:
        raise ValueError(f"Missing '{MTU_COL}' in {path}. Columns: {list(df.columns)}")

    # If the file contains multiple areas, keep HU if possible
    if "Area" in df.columns:
        hu_mask = df["Area"].astype(str).str.contains(r"\bHU\b|Hungary", case=False, na=False)
        if hu_mask.any():
            df = df.loc[hu_mask].copy()

    price_col = pick_price_col(df)

    # Parse interval boundaries
    starts, ends = zip(*df["MTU (CET/CEST)"].astype(str).map(parse_mtu_start_end))
    df["start"] = pd.DatetimeIndex(starts)
    df["end"]   = pd.DatetimeIndex(ends)

    # Localize to Budapest with DST handling:
    # - nonexistent times (spring forward): shift forward
    # - ambiguous (fall back): infer (works well when series is sorted/continuous)
    #df = df.sort_values("start_naive")
    # df["start"] = df["start_naive"].dt.tz_localize(TZ, ambiguous="infer", nonexistent="shift_forward")
    # df["end"]   = df["end_naive"].dt.tz_localize(TZ, ambiguous="infer", nonexistent="shift_forward")

    df["y"] = pd.to_numeric(df[price_col], errors="coerce")
    df = df.dropna(subset=["y"])

    out = df[["start", "end", "y"]].copy()
    out["source_file"] = path
    out["delta_min"] = (out["end"] - out["start"]).dt.total_seconds() / 60.0
    return out

In [17]:
# --- Load all files ---
parts = []
for f in FILES:
    try:
        parts.append(load_one(f))
    except FileNotFoundError:
        # ignore optional missing file(s)
        pass

raw = pd.concat(parts, ignore_index=True)
raw = raw.drop_duplicates(subset=["start"]).sort_values("start").reset_index(drop=True)
print("Loaded rows:", len(raw))
print("Date range:", raw["start"].min(), "→", raw["start"].max())
print("\nResolution (minutes) counts:")
print(raw["delta_min"].value_counts(dropna=False).head(10))

Loaded rows: 48840
Date range: 2023-09-16 00:00:00+02:00 → 2026-02-24 23:45:00+01:00

Resolution (minutes) counts:
delta_min
15.0    40032
60.0     8808
Name: count, dtype: int64


In [18]:
display(raw)

,start,end,y,source_file,delta_min
0,2023-09-16 00:00:00+02:00,2023-09-16 01:00:00+02:00,96.24,Entso_e\DA_price\GUI_ENERGY_PRICES_20230915220...,60.0
1,2023-09-16 01:00:00+02:00,2023-09-16 02:00:00+02:00,94.02,Entso_e\DA_price\GUI_ENERGY_PRICES_20230915220...,60.0
2,2023-09-16 02:00:00+02:00,2023-09-16 03:00:00+02:00,90.62,Entso_e\DA_price\GUI_ENERGY_PRICES_20230915220...,60.0
3,2023-09-16 03:00:00+02:00,2023-09-16 04:00:00+02:00,87.99,Entso_e\DA_price\GUI_ENERGY_PRICES_20230915220...,60.0
4,2023-09-16 04:00:00+02:00,2023-09-16 05:00:00+02:00,88.92,Entso_e\DA_price\GUI_ENERGY_PRICES_20230915220...,60.0
...,...,...,...,...,...
48835,2026-02-24 22:45:00+01:00,2026-02-24 23:00:00+01:00,106.19,Entso_e\DA_price\GUI_ENERGY_PRICES_20251231230...,15.0
48836,2026-02-24 23:00:00+01:00,2026-02-24 23:15:00+01:00,103.46,Entso_e\DA_price\GUI_ENERGY_PRICES_20251231230...,15.0
48837,2026-02-24 23:15:00+01:00,2026-02-24 23:30:00+01:00,101.93,Entso_e\DA_price\GUI_ENERGY_PRICES_20251231230...,15.0
48838,2026-02-24 23:30:00+01:00,2026-02-24 23:45:00+01:00,97.30,Entso_e\DA_price\GUI_ENERGY_PRICES_20251231230...,15.0


In [19]:
# --- Sanity check: are there mixed resolutions? ---
# 15-min expected: delta_min ~ 15
# hourly expected: delta_min ~ 60
raw["resolution"] = np.where(np.isclose(raw["delta_min"], 15), "15min",
                      np.where(np.isclose(raw["delta_min"], 60), "60min", "other"))

print("\nResolution labels:")
print(raw["resolution"].value_counts())


Resolution labels:
resolution
15min    40032
60min     8808
Name: count, dtype: int64


In [20]:
# --- Build a continuous 15-minute grid and mark observed points ---
# NOTE: If you truly have hourly points pre-2025-10-01, they will land on the 15-min grid
#       but only every 60 minutes; later Phase 2 will handle disaggregation.
ts = raw.set_index("start")[["y", "resolution", "source_file"]].sort_index()

full_index = pd.date_range(ts.index.min(), ts.index.max(), freq="15min", tz=TZ)
grid = ts.reindex(full_index)

grid.index.name = "ts"
grid["is_observed"] = grid["y"].notna()
# Calendar features (useful later; harmless now)
grid["qod"] = (grid.index.hour * 4 + (grid.index.minute // 15)).astype(int)  # 0..95 (mostly)
grid["dow"] = grid.index.dayofweek.astype(int)  # 0=Mon
grid["month"] = grid.index.month.astype(int)
grid["is_weekend"] = (grid["dow"] >= 5).astype(int)

# --- Diagnostics ---
# Missingness overall
missing_ratio = 1.0 - grid["is_observed"].mean()
print("\nMissing ratio on full 15-min grid:", round(missing_ratio, 4))


Missing ratio on full 15-min grid: 0.4303


In [21]:
# Check for DST-day anomalies: count quarters per day
daily_counts = grid["is_observed"].groupby(grid.index.normalize()).sum()
print("\nObserved 15-min points per day (sample extremes):")
print("min:", int(daily_counts.min()), "max:", int(daily_counts.max()))
print(daily_counts.value_counts().sort_index().head(10))


Observed 15-min points per day (sample extremes):
min: 0 max: 100
is_observed
0      109
23       1
24     365
25       1
92       1
96     415
100      1
Name: count, dtype: int64


In [22]:
# Identify days that are not 96 (could be DST days or gaps)
odd_days = daily_counts[(daily_counts != 96) & (daily_counts != 0)]
print("\nDays with observed count not equal to 96 (show up to 15):")
print(odd_days.head(15))


Days with observed count not equal to 96 (show up to 15):
ts
2023-09-16 00:00:00+02:00    24
2024-01-01 00:00:00+01:00    24
2024-01-02 00:00:00+01:00    24
2024-01-03 00:00:00+01:00    24
2024-01-04 00:00:00+01:00    24
2024-01-05 00:00:00+01:00    24
2024-01-06 00:00:00+01:00    24
2024-01-07 00:00:00+01:00    24
2024-01-08 00:00:00+01:00    24
2024-01-09 00:00:00+01:00    24
2024-01-10 00:00:00+01:00    24
2024-01-11 00:00:00+01:00    24
2024-01-12 00:00:00+01:00    24
2024-01-13 00:00:00+01:00    24
2024-01-14 00:00:00+01:00    24
Name: is_observed, dtype: int64


In [23]:
# Quick look around 2025-10-01 (structural breakpoint you mentioned)
breakpoint = pd.Timestamp("2025-10-01", tz=TZ)
window = grid.loc[breakpoint - pd.Timedelta(days=3): breakpoint + pd.Timedelta(days=3)]
print("\nResolution labels around 2025-10-01 (observed rows only):")
print(window.loc[window["is_observed"], "resolution"].value_counts(dropna=False))


Resolution labels around 2025-10-01 (observed rows only):
resolution
15min    577
Name: count, dtype: int64


In [24]:
# Save Phase 1 outputs for Phase 2
# - raw: original observations (mixed resolution)
# - grid: full 15-min grid with observed flags (hourly points appear sparsely)
PHASE1_RAW = raw
PHASE1_GRID = grid

print("\nPhase 1 complete. Objects in memory: PHASE1_RAW, PHASE1_GRID")


Phase 1 complete. Objects in memory: PHASE1_RAW, PHASE1_GRID


In [25]:
display(PHASE1_RAW)

,start,end,y,source_file,delta_min,resolution
0,2023-09-16 00:00:00+02:00,2023-09-16 01:00:00+02:00,96.24,Entso_e\DA_price\GUI_ENERGY_PRICES_20230915220...,60.0,60min
1,2023-09-16 01:00:00+02:00,2023-09-16 02:00:00+02:00,94.02,Entso_e\DA_price\GUI_ENERGY_PRICES_20230915220...,60.0,60min
2,2023-09-16 02:00:00+02:00,2023-09-16 03:00:00+02:00,90.62,Entso_e\DA_price\GUI_ENERGY_PRICES_20230915220...,60.0,60min
3,2023-09-16 03:00:00+02:00,2023-09-16 04:00:00+02:00,87.99,Entso_e\DA_price\GUI_ENERGY_PRICES_20230915220...,60.0,60min
4,2023-09-16 04:00:00+02:00,2023-09-16 05:00:00+02:00,88.92,Entso_e\DA_price\GUI_ENERGY_PRICES_20230915220...,60.0,60min
...,...,...,...,...,...,...
48835,2026-02-24 22:45:00+01:00,2026-02-24 23:00:00+01:00,106.19,Entso_e\DA_price\GUI_ENERGY_PRICES_20251231230...,15.0,15min
48836,2026-02-24 23:00:00+01:00,2026-02-24 23:15:00+01:00,103.46,Entso_e\DA_price\GUI_ENERGY_PRICES_20251231230...,15.0,15min
48837,2026-02-24 23:15:00+01:00,2026-02-24 23:30:00+01:00,101.93,Entso_e\DA_price\GUI_ENERGY_PRICES_20251231230...,15.0,15min
48838,2026-02-24 23:30:00+01:00,2026-02-24 23:45:00+01:00,97.30,Entso_e\DA_price\GUI_ENERGY_PRICES_20251231230...,15.0,15min


In [26]:
display(PHASE1_GRID)

,y,resolution,source_file,is_observed,qod,dow,month,is_weekend
ts,,,,,,,,
2023-09-16 00:00:00+02:00,96.24,60min,Entso_e\DA_price\GUI_ENERGY_PRICES_20230915220...,True,0,5,9,1
2023-09-16 00:15:00+02:00,NaN,NaN,NaN,False,1,5,9,1
2023-09-16 00:30:00+02:00,NaN,NaN,NaN,False,2,5,9,1
2023-09-16 00:45:00+02:00,NaN,NaN,NaN,False,3,5,9,1
2023-09-16 01:00:00+02:00,94.02,60min,Entso_e\DA_price\GUI_ENERGY_PRICES_20230915220...,True,4,5,9,1
...,...,...,...,...,...,...,...,...
2026-02-24 22:45:00+01:00,106.19,15min,Entso_e\DA_price\GUI_ENERGY_PRICES_20251231230...,True,91,1,2,0
2026-02-24 23:00:00+01:00,103.46,15min,Entso_e\DA_price\GUI_ENERGY_PRICES_20251231230...,True,92,1,2,0
2026-02-24 23:15:00+01:00,101.93,15min,Entso_e\DA_price\GUI_ENERGY_PRICES_20251231230...,True,93,1,2,0


In [27]:

BREAKPOINT = pd.Timestamp("2025-10-01", tz=TZ)

# Choose evaluation window length
TEST_DAYS = 30          # last 60 days of the REAL_15M segment
MIN_TRAIN_DAYS = 90    # require at least ~6 months training inside REAL_15M

# Model choice: LightGBM if available, otherwise sklearn fallback
USE_QUANTILES = False   # set True if you want P10/P50/P90 later (more code)

In [28]:
# -------------------------------
# 0) Extract TRUE 15-min segment (post-breakpoint) from Phase 1 grid
# -------------------------------
g = PHASE1_GRID.copy()

# Keep only observed points
g = g[g["is_observed"]].copy()

# Identify true 15-min portion (resolution label might be missing depending on your Phase 1;
# if you have it, use it; otherwise infer from spacing by resampling completeness)
if "resolution" in g.columns:
    g15 = g[(g.index >= BREAKPOINT) & (g["resolution"] == "15min")].copy()
else:
    # Fallback: assume post-breakpoint is 15-min if present; still require dense days
    g15 = g[g.index >= BREAKPOINT].copy()

# Basic sanity: require enough data
if g15.empty:
    raise ValueError("No observed 15-min data found after breakpoint. Check Phase 1 output and breakpoint.")

# Ensure sorted and regular-ish
g15 = g15.sort_index()
g15 = g15[["y"]].copy()

In [29]:
# -------------------------------
# 1) Build simulated hourly series from TRUE 15-min
# -------------------------------
# Hourly mean (you could also use weighted or other definition depending on market)
hour = g15["y"].resample("1H").mean().to_frame("y_hour")

# Add hourly calendar + ramp features (all available in hourly-only world)
hour["hod"] = hour.index.hour.astype(int)
hour["dow"] = hour.index.dayofweek.astype(int)
hour["month"] = hour.index.month.astype(int)
hour["is_weekend"] = (hour["dow"] >= 5).astype(int)

hour["hour_ramp_1"] = hour["y_hour"].diff(1)     # hour-to-hour ramp
hour["hour_ramp_24"] = hour["y_hour"].diff(24)   # day-to-day ramp at same hour

C:\Users\local_user\AppData\Local\Temp\ipykernel_3268\123186266.py:5: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hour = g15["y"].resample("1H").mean().to_frame("y_hour")


In [30]:
# -------------------------------
# 2) Align hourly info back to each 15-min point
# -------------------------------
df = g15.copy()
print(df)
utc_index = df.index.tz_convert("UTC")
hour_utc = utc_index.floor("H")
df["hour_ts"] = hour_utc.tz_convert("Europe/Budapest")

# Merge hourly-only features
df = df.join(hour, on="hour_ts", how="left")

# Quarter-in-hour (0..3)
df["q_in_hour"] = (df.index.minute // 15).astype(int)

# Quarter-of-day (0..95)
df["qod"] = (df.index.hour * 4 + df["q_in_hour"]).astype(int)

# Target: deviation from hourly mean (this is what disaggregation learns)
df["delta_true"] = df["y"] - df["y_hour"]

# Drop rows where hourly aggregation missing (rare, but possible with gaps)
df = df.dropna(subset=["y", "y_hour", "delta_true"])

                                y
ts                               
2025-10-01 00:00:00+02:00  108.98
2025-10-01 00:15:00+02:00   97.08
2025-10-01 00:30:00+02:00   90.97
2025-10-01 00:45:00+02:00   90.60
2025-10-01 01:00:00+02:00   91.15
...                           ...
2026-02-24 22:45:00+01:00  106.19
2026-02-24 23:00:00+01:00  103.46
2026-02-24 23:15:00+01:00  101.93
2026-02-24 23:30:00+01:00   97.30
2026-02-24 23:45:00+01:00   93.69

[13828 rows x 1 columns]


C:\Users\local_user\AppData\Local\Temp\ipykernel_3268\533946394.py:7: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hour_utc = utc_index.floor("H")


In [31]:
# -------------------------------
# 3) Create time-based train/test split (block split)
# -------------------------------
# We split by DAY to avoid leakage across intra-day adjacency.
df["day"] = df.index.normalize()

all_days = np.array(sorted(df["day"].unique()))
if len(all_days) < (MIN_TRAIN_DAYS + TEST_DAYS):
    raise ValueError(
        f"Not enough post-breakpoint days for robust validation. "
        f"Have {len(all_days)} days, need at least {MIN_TRAIN_DAYS + TEST_DAYS}."
    )

test_days = all_days[-TEST_DAYS:]
train_days = all_days[:-TEST_DAYS]

train = df[df["day"].isin(train_days)].copy()
test  = df[df["day"].isin(test_days)].copy()

In [32]:
# -------------------------------
# 4) Train a disaggregation model: predict delta_true from hourly-only features + quarter position
# -------------------------------
FEATURES = [
    "y_hour",
    "hour_ramp_1",
    "hour_ramp_24",
    "hod",
    "dow",
    "month",
    "is_weekend",
    "q_in_hour",
]

X_train = train[FEATURES]
y_train = train["delta_true"]

X_test  = test[FEATURES]
y_test  = test["delta_true"]

# Model: LightGBM if available, else sklearn GradientBoostingRegressor
model_name = None
model = None

try:
    import lightgbm as lgb
    model_name = "LightGBM"
    model = lgb.LGBMRegressor(
        n_estimators=1200,
        learning_rate=0.03,
        num_leaves=64,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        n_jobs=-1,
    )
except Exception:
    from sklearn.ensemble import HistGradientBoostingRegressor
    model_name = "HistGradientBoostingRegressor"
    model = HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=8,
        max_iter=600,
        random_state=42
    )

model.fit(X_train, y_train)

# Predict deltas, reconstruct 15-min prices
test = test.copy()
test["delta_hat"] = model.predict(X_test)
test["y_hat"] = test["y_hour"] + test["delta_hat"]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000288 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 807
[LightGBM] [Info] Number of data points in the train set: 10948, number of used features: 8
[LightGBM] [Info] Start training from score -0.000000


In [34]:
test

,y,hour_ts,y_hour,hod,dow,month,is_weekend,hour_ramp_1,hour_ramp_24,q_in_hour,qod,delta_true,day,delta_hat,y_hat,abs_err
ts,,,,,,,,,,,,,,,,
2026-01-23 00:00:00+01:00,148.99,2026-01-23 00:00:00+01:00,147.9075,0,4,1,0,5.6875,20.3350,0,0,1.0825,2026-01-23 00:00:00+01:00,8.614135,156.521635,7.531635
2026-01-23 00:15:00+01:00,150.55,2026-01-23 00:00:00+01:00,147.9075,0,4,1,0,5.6875,20.3350,1,1,2.6425,2026-01-23 00:00:00+01:00,2.540356,150.447856,0.102144
2026-01-23 00:30:00+01:00,147.82,2026-01-23 00:00:00+01:00,147.9075,0,4,1,0,5.6875,20.3350,2,2,-0.0875,2026-01-23 00:00:00+01:00,-1.003925,146.903575,0.916425
2026-01-23 00:45:00+01:00,144.27,2026-01-23 00:00:00+01:00,147.9075,0,4,1,0,5.6875,20.3350,3,3,-3.6375,2026-01-23 00:00:00+01:00,-9.484610,138.422890,5.847110
2026-01-23 01:00:00+01:00,141.84,2026-01-23 01:00:00+01:00,144.9775,1,4,1,0,-2.9300,17.9200,0,4,-3.1375,2026-01-23 00:00:00+01:00,5.151205,150.128705,8.288705
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-24 22:45:00+01:00,106.19,2026-02-24 22:00:00+01:00,110.7500,22,1,2,0,-7.1825,5.8325,3,91,-4.5600,2026-02-24 00:00:00+01:00,-10.387123,100.362877,5.827123
2026-02-24 23:00:00+01:00,103.46,2026-02-24 23:00:00+01:00,99.0950,23,1,2,0,-11.6550,7.2150,0,92,4.3650,2026-02-24 00:00:00+01:00,11.154704,110.249704,6.789704
2026-02-24 23:15:00+01:00,101.93,2026-02-24 23:00:00+01:00,99.0950,23,1,2,0,-11.6550,7.2150,1,93,2.8350,2026-02-24 00:00:00+01:00,2.588262,101.683262,0.246738


In [33]:
# -------------------------------
# 5) Evaluate reconstruction quality (THIS answers "can synthetic be created?")
# -------------------------------
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(test["y"], test["y_hat"])
rmse = float(np.sqrt(mean_squared_error(test["y"], test["y_hat"])))

# Quarter-wise errors
test["abs_err"] = (test["y"] - test["y_hat"]).abs()
mae_by_q = test.groupby("q_in_hour")["abs_err"].mean().rename("MAE")

# Spike subset: evaluate on top 1% true prices (heavy-tail behavior)
thr = test["y"].quantile(0.99)
spike = test[test["y"] >= thr]
spike_mae = mean_absolute_error(spike["y"], spike["y_hat"]) if len(spike) else np.nan
spike_rmse = float(np.sqrt(mean_squared_error(spike["y"], spike["y_hat"]))) if len(spike) else np.nan

# Distribution check (quantiles)
q_levels = [0.01, 0.05, 0.5, 0.95, 0.99]
true_q = test["y"].quantile(q_levels)
hat_q  = test["y_hat"].quantile(q_levels)
quantile_compare = pd.DataFrame({"true": true_q, "recon": hat_q})
quantile_compare.index.name = "quantile"

print("\n============================")
print("PHASE 2: Proxy reconstruction validation (hourly → 15-min)")
print("============================")
print(f"Model: {model_name}")
print(f"Train days: {train_days[0].date()} → {train_days[-1].date()}  ({len(train_days)} days)")
print(f"Test  days: {test_days[0].date()} → {test_days[-1].date()}   ({len(test_days)} days)")
print(f"\nOverall MAE : {mae:.4f}")
print(f"Overall RMSE: {rmse:.4f}")
print("\nMAE by quarter-in-hour (0..3):")
print(mae_by_q)

print("\nSpike subset (top 1% of TRUE prices in test):")
print(f"Threshold (99th pct): {thr:.3f}")
print(f"Spike count: {len(spike)}")
print(f"Spike MAE : {spike_mae:.4f}")
print(f"Spike RMSE: {spike_rmse:.4f}")

print("\nQuantile comparison (TRUE vs reconstructed):")
print(quantile_compare)

# Optional: error by quarter-of-day (shape quality across day)
mae_by_qod = test.groupby("qod")["abs_err"].mean()
print("\nMAE by quarter-of-day computed (length):", len(mae_by_qod))


PHASE 2: Proxy reconstruction validation (hourly → 15-min)
Model: LightGBM
Train days: 2025-10-01 → 2026-01-22  (114 days)
Test  days: 2026-01-23 → 2026-02-24   (30 days)

Overall MAE : 4.3907
Overall RMSE: 9.3671

MAE by quarter-in-hour (0..3):
q_in_hour
0    4.732806
1    3.637123
2    3.090046
3    6.102671
Name: MAE, dtype: float64

Spike subset (top 1% of TRUE prices in test):
Threshold (99th pct): 287.962
Spike count: 29
Spike MAE : 45.1027
Spike RMSE: 58.3555

Quantile comparison (TRUE vs reconstructed):
              true       recon
quantile                      
0.01       49.9537   53.128441
0.05       76.4675   74.260075
0.50      121.4450  121.629941
0.95      207.7965  209.489189
0.99      287.9619  283.414042

MAE by quarter-of-day computed (length): 96


In [35]:
# -----------------------
# Helper: DST-safe hour bucket key for 15-min timestamps
# -----------------------
def hour_bucket_local(idx: pd.DatetimeIndex, tz: str = TZ) -> pd.DatetimeIndex:
    """Return hour bucket timestamps in local tz, computed by flooring in UTC (DST-safe)."""
    return idx.tz_convert("UTC").floor("H").tz_convert(tz)

In [36]:
# -----------------------
# 1) Build training set from TRUE 15-min data (post-breakpoint)
# -----------------------
g = PHASE1_GRID.copy()
g = g[g["is_observed"]].copy()

# Prefer the true 15-min post-breakpoint part
if "resolution" in g.columns:
    g15 = g[(g.index >= BREAKPOINT) & (g["resolution"] == "15min")].copy()
else:
    g15 = g[g.index >= BREAKPOINT].copy()

g15 = g15.sort_index()
g15 = g15[["y"]].dropna()

if g15.empty:
    raise ValueError("No observed 15-min data found after 2025-10-01. Check Phase 1 grid and breakpoint.")

# Hourly mean series derived from TRUE 15-min (simulated hourly-only)
hour = g15["y"].resample("1H").mean().to_frame("y_hour")
hour["hod"] = hour.index.hour.astype(int)
hour["dow"] = hour.index.dayofweek.astype(int)
hour["month"] = hour.index.month.astype(int)
hour["is_weekend"] = (hour["dow"] >= 5).astype(int)
hour["hour_ramp_1"] = hour["y_hour"].diff(1)
hour["hour_ramp_24"] = hour["y_hour"].diff(24)

# Align hourly-only info to each 15-min timestamp
train_df = g15.copy()
train_df["hour_ts"] = hour_bucket_local(train_df.index)  # DST-safe hour key
train_df = train_df.join(hour, on="hour_ts", how="left")

train_df["q_in_hour"] = (train_df.index.minute // 15).astype(int)  # 0..3
train_df["delta_true"] = train_df["y"] - train_df["y_hour"]

FEATURES = [
    "y_hour", "hour_ramp_1", "hour_ramp_24",
    "hod", "dow", "month", "is_weekend",
    "q_in_hour",
]

train_df = train_df.dropna(subset=FEATURES + ["delta_true"]).copy()

C:\Users\local_user\AppData\Local\Temp\ipykernel_3268\1204328834.py:20: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hour = g15["y"].resample("1H").mean().to_frame("y_hour")
C:\Users\local_user\AppData\Local\Temp\ipykernel_3268\915592914.py:6: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  return idx.tz_convert("UTC").floor("H").tz_convert(tz)


In [37]:
# -----------------------
# 2) Train quantile models for delta (P10, P50, P90)
# -----------------------
import lightgbm as lgb

def fit_quantile(alpha: float):
    m = lgb.LGBMRegressor(
        objective="quantile",
        alpha=alpha,
        n_estimators=1400,
        learning_rate=0.03,
        num_leaves=64,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        n_jobs=-1,
    )
    m.fit(train_df[FEATURES], train_df["delta_true"])
    return m

q10_model = fit_quantile(0.10)
q50_model = fit_quantile(0.50)  # use as point estimate (robust)
q90_model = fit_quantile(0.90)

PHASE3_QMODELS = {"p10": q10_model, "p50": q50_model, "p90": q90_model}
PHASE3_FEATURES = FEATURES

print("Phase 3: trained quantile disaggregation models on post-breakpoint TRUE 15-min data.")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001933 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 808
[LightGBM] [Info] Number of data points in the train set: 13636, number of used features: 8
[LightGBM] [Info] Start training from score -14.057500
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000121 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 808
[LightGBM] [Info] Number of data points in the train set: 13636, number of used features: 8
[LightGBM] [Info] Start training from score -0.095000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000122 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bin

In [38]:
# -----------------------
# 3) Extract PRE-breakpoint hourly observations (the thing we backcast from)
# -----------------------
raw = PHASE1_RAW.copy()

# Determine hourly rows: delta_min ~ 60 and start < breakpoint
if "delta_min" not in raw.columns:
    raw["delta_min"] = (raw["end"] - raw["start"]).dt.total_seconds() / 60.0

pre_hour = raw[(raw["start"] < BREAKPOINT) & (np.isclose(raw["delta_min"], 60.0))].copy()
pre_hour = pre_hour.sort_values("start")

if pre_hour.empty:
    raise ValueError(
        "No pre-breakpoint hourly observations found (delta_min≈60). "
        "If your pre-breakpoint data is not hourly in PHASE1_RAW, check Phase 1 ingestion."
    )

# Create hourly series
pre_hour = pre_hour.set_index("start")[["y"]].rename(columns={"y": "y_hour"})
pre_hour.index.name = "hour_ts"

# Build a continuous hourly index (DST-safe): do in UTC then convert back
hour_utc = pd.date_range(
    pre_hour.index.min().tz_convert("UTC"),
    pre_hour.index.max().tz_convert("UTC"),
    freq="1H",
    tz="UTC",
)
hour_local = hour_utc.tz_convert(TZ)

pre_hour = pre_hour.reindex(hour_local)
pre_hour["is_hour_observed"] = pre_hour["y_hour"].notna()

# For backcasting we typically need all hours; if there are gaps, we can:
# - leave them NaN (synthetic will be NaN for those hours)
# - or interpolate hourly. For publication safety, default to NO interpolation.
# If you want interpolation, do it explicitly and document it.

# Hourly features
pre_hour["hod"] = pre_hour.index.hour.astype(int)
pre_hour["dow"] = pre_hour.index.dayofweek.astype(int)
pre_hour["month"] = pre_hour.index.month.astype(int)
pre_hour["is_weekend"] = (pre_hour["dow"] >= 5).astype(int)
pre_hour["hour_ramp_1"] = pre_hour["y_hour"].diff(1)
pre_hour["hour_ramp_24"] = pre_hour["y_hour"].diff(24)

C:\Users\local_user\AppData\Local\Temp\ipykernel_3268\1296502431.py:24: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hour_utc = pd.date_range(


In [39]:
# -----------------------
# 4) Expand each hour into 4 quarters (DST-safe, generate in UTC then convert)
# -----------------------
# Create quarter timestamps for each hour in UTC, then convert to local tz
pre_hour_utc = pre_hour.index.tz_convert("UTC")

quarters_utc = []
q_in_hour = []
hour_key_utc = []

for q, minutes in enumerate([0, 15, 30, 45]):
    quarters_utc.append(pre_hour_utc + pd.Timedelta(minutes=minutes))
    q_in_hour.append(np.full(len(pre_hour_utc), q, dtype=int))
    hour_key_utc.append(pre_hour_utc)  # hour bucket key in UTC

quarters_utc = pd.DatetimeIndex(np.concatenate(quarters_utc))
q_in_hour = np.concatenate(q_in_hour)
hour_key_utc = pd.DatetimeIndex(np.concatenate(hour_key_utc))

synthetic = pd.DataFrame({
    "ts_utc": quarters_utc,
    "hour_ts_utc": hour_key_utc,
    "q_in_hour": q_in_hour
}).sort_values("ts_utc")

# Convert to local tz for final index
synthetic["ts"] = synthetic["ts_utc"].dt.tz_convert(TZ)
synthetic["hour_ts"] = synthetic["hour_ts_utc"].dt.tz_convert(TZ)

# Join hourly features/price onto each quarter
synthetic = synthetic.join(pre_hour, on="hour_ts", how="left")

# Prepare feature matrix and predict delta quantiles
X_syn = synthetic[FEATURES]

# Predict deltas (will be NaN where hourly y_hour is NaN)
synthetic["delta_p10"] = np.where(
    synthetic["y_hour"].notna(),
    q10_model.predict(X_syn),
    np.nan
)
synthetic["delta_p50"] = np.where(
    synthetic["y_hour"].notna(),
    q50_model.predict(X_syn),
    np.nan
)
synthetic["delta_p90"] = np.where(
    synthetic["y_hour"].notna(),
    q90_model.predict(X_syn),
    np.nan
)

# Reconstruct price quantiles
synthetic["y_p10"] = synthetic["y_hour"] + synthetic["delta_p10"]
synthetic["y_p50"] = synthetic["y_hour"] + synthetic["delta_p50"]  # point estimate
synthetic["y_p90"] = synthetic["y_hour"] + synthetic["delta_p90"]

synthetic["is_synthetic"] = 1
synthetic["is_observed"] = 0

# Keep clean output
SYNTH_15M_PRE = synthetic.set_index("ts")[["y_p10", "y_p50", "y_p90", "is_synthetic", "is_observed"]].sort_index()

print("Phase 3: generated pre-breakpoint synthetic 15-min series with P10/P50/P90.")
print("Synthetic range:", SYNTH_15M_PRE.index.min(), "→", SYNTH_15M_PRE.index.max())
print("Synthetic rows:", len(SYNTH_15M_PRE))

Phase 3: generated pre-breakpoint synthetic 15-min series with P10/P50/P90.
Synthetic range: 2023-09-16 00:00:00+02:00 → 2024-12-31 23:45:00+01:00
Synthetic rows: 45412


In [40]:
SYNTH_15M_PRE

,y_p10,y_p50,y_p90,is_synthetic,is_observed
ts,,,,,
2023-09-16 00:00:00+02:00,95.314204,103.433876,109.537610,1,0
2023-09-16 00:15:00+02:00,94.322916,98.290144,101.788504,1,0
2023-09-16 00:30:00+02:00,89.233976,93.182986,98.338415,1,0
2023-09-16 00:45:00+02:00,82.512575,91.060511,96.282437,1,0
2023-09-16 01:00:00+02:00,93.716256,98.318248,102.421527,1,0
...,...,...,...,...,...
2024-12-31 22:45:00+01:00,108.266769,113.192800,120.973647,1,0
2024-12-31 23:00:00+01:00,126.477141,133.574267,142.591483,1,0
2024-12-31 23:15:00+01:00,121.867305,126.870481,132.257795,1,0


In [41]:
# -----------------------
# 5) (Optional) Build unified 15-min dataset: pre synthetic + post real
# -----------------------
# Use P50 as the single point price for synthetic history.
POST_REAL_15M = PHASE1_GRID.loc[PHASE1_GRID["is_observed"] & (PHASE1_GRID.index >= BREAKPOINT), ["y"]].copy()
POST_REAL_15M = POST_REAL_15M.rename(columns={"y": "y_real"})
POST_REAL_15M["is_synthetic"] = 0
POST_REAL_15M["is_observed"] = 1

# Create a unified point series "y_point" (synthetic uses y_p50; real uses y_real)
pre_point = SYNTH_15M_PRE.copy()
pre_point["y_point"] = pre_point["y_p50"]

post_point = POST_REAL_15M.copy()
post_point["y_point"] = post_point["y_real"]
post_point["y_p10"] = np.nan
post_point["y_p50"] = np.nan
post_point["y_p90"] = np.nan

UNIFIED_15M = pd.concat([
    pre_point[["y_point", "y_p10", "y_p50", "y_p90", "is_synthetic", "is_observed"]],
    post_point[["y_point", "y_p10", "y_p50", "y_p90", "is_synthetic", "is_observed"]],
]).sort_index()

PHASE3_SYNTH_15M_PRE = SYNTH_15M_PRE
PHASE3_UNIFIED_15M = UNIFIED_15M

print("\nPhase 3 complete. Objects in memory:")
print(" - PHASE3_QMODELS (dict of quantile models)")
print(" - PHASE3_SYNTH_15M_PRE (pre-breakpoint synthetic 15-min with P10/P50/P90)")
print(" - PHASE3_UNIFIED_15M (unified 15-min point series + bands where available)")


Phase 3 complete. Objects in memory:
 - PHASE3_QMODELS (dict of quantile models)
 - PHASE3_SYNTH_15M_PRE (pre-breakpoint synthetic 15-min with P10/P50/P90)
 - PHASE3_UNIFIED_15M (unified 15-min point series + bands where available)


In [80]:
# ----------------------------
# User settings (adjust here)
# ----------------------------
SYNTH_WEIGHT = 0.3          # Model2: weight for synthetic samples (REAL weight = 1.0)
RETRAIN_EVERY = 1           # retrain models every N forecast days (1 = daily retrain)
TEST_DAYS = 30              # number of REAL days at the end to evaluate with rolling-origin
VAL_DAYS = 14               # optional validation window inside training for sanity-check metrics (not used for early stop here)

# Feature lags in 15-min steps
STATE_LAGS = [1, 4, 8, 24, 96, 192, 672]   # 15m, 1h, 2h, 6h, 1d, 2d, 1w
STATE_ROLL_WINS = [24, 96, 672]            # rolling windows on past y (6h, 1d, 1w)

# LightGBM params (good starting defaults)
LGB_PARAMS = dict(
    n_estimators=1500,
    learning_rate=0.03,
    num_leaves=64,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
)

LOAD_FILES = [
    r"Entso_e\Load\GUI_TOTAL_LOAD_DAYAHEAD_202212312300-202312312300.csv",
    r"Entso_e\Load\GUI_TOTAL_LOAD_DAYAHEAD_202312312300-202412312300.csv",
    r"Entso_e\Load\GUI_TOTAL_LOAD_DAYAHEAD_202412312300-202512312300.csv",
    r"Entso_e\Load\GUI_TOTAL_LOAD_DAYAHEAD_202512312300-202612312300.csv",
]

MTU_COL = "MTU (CET/CEST)"

In [88]:
# ----------------------------
# Load parsing helpers (CET/CEST label -> unambiguous instant)
# ----------------------------
def parse_dt_with_label(s: str) -> pd.Timestamp:
    """
    Input: '31/03/2024 01:00:00 (CET)' or '(CEST)'
    Return: tz-aware timestamp in Europe/Budapest (as an instant).
    """
    s = s.strip()
    m = re.match(r"^(.*?)(?:\s+\((CET|CEST)\))?$", s)
    if not m:
        raise ValueError(f"Unrecognized datetime format: {s}")
    dt_part = m.group(1).strip()
    label = m.group(2)

    dt = pd.to_datetime(dt_part, format="%d/%m/%Y %H:%M", errors="raise")

    if label is None:
        # fallback: attach TZ conservatively (rare for ENTSO-E exports)
        return dt.tz_localize(TZ, ambiguous="NaT", nonexistent="shift_forward")

    offset_hours = 1 if label == "CET" else 2  # CET=UTC+1, CEST=UTC+2
    utc = (dt - pd.Timedelta(hours=offset_hours)).tz_localize("UTC")
    return utc.tz_convert(TZ)

def parse_mtu_start(mtu: str) -> pd.Timestamp:
    # "dd/mm/YYYY HH:MM:SS (CET) - dd/mm/YYYY HH:MM:SS (CET)"
    a = mtu.split("-")[0].strip()
    return parse_dt_with_label(a)

def pick_load_col(df: pd.DataFrame) -> str:
    # Try common ENTSO-E export column names
    print(df)
    candidates = [
        "Total Load Forecast (MW)",
        "Total Load Forecast (MWh)",
        "Total Load Forecast (MW )",
        "Total Load (MW)",
        "Value",
    ]
    for c in candidates:
        if c in df.columns:
            return c
    # fallback: pick first numeric-like column not MTU/Area
    non = {MTU_COL, "Area", "Bidding Zone", "Country"}
    for c in df.columns:
        if c in non: 
            continue
        if df[c].dtype != object:
            return c
    raise ValueError(f"Could not identify load column. Columns: {list(df.columns)}")

def load_total_load_forecast(paths) -> pd.Series:
    parts = []
    for p in paths:
        df = pd.read_csv(p)
        if MTU_COL not in df.columns:
            raise ValueError(f"Missing '{MTU_COL}' in {p}")

        # Filter HU if applicable
        if "Area" in df.columns:
            hu_mask = df["Area"].astype(str).str.contains(r"\bHU\b|Hungary", case=False, na=False)
            if hu_mask.any():
                df = df.loc[hu_mask].copy()

        col = "Day-ahead Total Load Forecast (MW)"

        ts = df[MTU_COL].astype(str).map(parse_mtu_start)
        vals = pd.to_numeric(df[col], errors="coerce")

        tmp = pd.DataFrame({"ts": ts, "load_fc": vals}).dropna()
        parts.append(tmp)

    all_df = pd.concat(parts, ignore_index=True).drop_duplicates(subset=["ts"]).sort_values("ts")
    s = all_df.set_index("ts")["load_fc"].sort_index()
    return s

In [89]:
load_fc = load_total_load_forecast(LOAD_FILES)
print("Loaded load forecast:", load_fc.index.min(), "→", load_fc.index.max(), "rows:", len(load_fc))

Loaded load forecast: 2023-01-01 00:00:00+01:00 → 2026-03-02 23:45:00+01:00 rows: 111072


In [92]:
# ----------------------------
# 1) Prepare unified series
# ----------------------------
u = PHASE3_UNIFIED_15M.copy().sort_index()

need_cols = {"y_point", "is_synthetic", "is_observed"}
missing = need_cols - set(u.columns)
if missing:
    raise ValueError(f"PHASE3_UNIFIED_15M missing columns: {missing}")
u["load_fc"] = load_fc.reindex(u.index)
# Ensure index tz-aware in Budapest
if u.index.tz is None:
    raise ValueError("PHASE3_UNIFIED_15M index must be tz-aware (Europe/Budapest).")


In [67]:
# ----------------------------
# 1) Build "daily cutoff table": one row per day with state features at t0 = day-1 23:45
#    and targets: the 96 prices for that day
# ----------------------------
def day_horizon_index(day: pd.Timestamp) -> pd.DatetimeIndex:
    """Return the 96 timestamps for a given local day (00:00..23:45)."""
    start = pd.Timestamp(day)
    return pd.date_range(start, start + pd.Timedelta(hours=24) - pd.Timedelta(minutes=15),
                         freq="15min", tz=TZ)

In [93]:
def compute_state_features_at_t0(series: pd.Series, t0: pd.Timestamp) -> dict:
    """
    Compute state features using only history up to and including t0.
    series: y_point indexed by 15-min timestamps
    """
    feats = {}
    y0 = series.loc[:t0]

    feats["last_y"] = float(y0.iloc[-1])

    # Lags at cutoff
    for L in STATE_LAGS:
        key = f"lag_{L}_t0"
        feats[key] = float(y0.iloc[-(L+1)])  # because y0.iloc[-1] is t0

    # Ramps at cutoff (examples)
    feats["ramp_1h_t0"] = feats["last_y"] - feats["lag_4_t0"]
    feats["ramp_6h_t0"] = feats["last_y"] - feats["lag_24_t0"]
    feats["ramp_1d_t0"] = feats["last_y"] - feats["lag_96_t0"]

    # Rolling stats ending at t0 (past values including t0)
    yvals = y0.values
    for w in STATE_ROLL_WINS:
        window = yvals[-w:]
        feats[f"roll_mean_{w}_t0"] = float(np.mean(window))
        feats[f"roll_std_{w}_t0"] = float(np.std(window, ddof=1)) if w > 1 else 0.0

    return feats

In [95]:
def build_daily_dataset(u_df: pd.DataFrame) -> pd.DataFrame:
    y = u_df["y_point"]
    lf = u_df["load_fc"]

    days = np.array(sorted(u_df.index.normalize().unique()))

    rows = []
    max_lag = max(STATE_LAGS)

    for day in days:
        day = pd.Timestamp(day)  # keep tz as-is
        h_idx = day_horizon_index(day)

        # bounds check
        if h_idx.min() < u_df.index.min() or h_idx.max() > u_df.index.max():
            continue

        # require targets and load available for all 96 quarters
        if y.loc[h_idx].isna().any():
            continue
        if lf.loc[h_idx].isna().any():
            continue

        t0 = (pd.Timestamp(day) - pd.Timedelta(minutes=15))
        if t0 not in y.index:
            continue

        hist_slice = y.loc[:t0]
        if len(hist_slice) < (max_lag + 2):
            continue

        state = compute_state_features_at_t0(y, t0)

        # Day-level load summary (same across 96 horizons; safe because it's forecast)
        load_day = lf.loc[h_idx]
        load_day_mean = float(load_day.mean())
        load_day_max = float(load_day.max())
        load_day_min = float(load_day.min())

        day_flags = u_df.loc[h_idx, ["is_synthetic", "is_observed"]].copy()

        # per-horizon rows
        for h, ts in enumerate(h_idx, start=1):
            r = dict(state)

            # horizon descriptors
            r["h"] = h
            r["q_in_hour_target"] = int(ts.minute // 15)
            r["qod_target"] = int(ts.hour * 4 + (ts.minute // 15))
            r["hod_target"] = int(ts.hour)
            r["dow_target"] = int(ts.dayofweek)
            r["month_target"] = int(ts.month)
            r["is_weekend_target"] = int(ts.dayofweek >= 5)

            # load forecast features (vary by horizon)
            r["load_fc_target"] = float(lf.loc[ts])
            # within-day load ramps (safe; forecast curve is known)
            # use past within-forecast-day values, which are still known because the entire forecast curve is known
            prev_1h_ts = ts - pd.Timedelta(hours=1)
            prev_6h_ts = ts - pd.Timedelta(hours=6)
            r["load_ramp_1h_target"] = float(lf.loc[ts] - (lf.loc[prev_1h_ts] if prev_1h_ts in lf.index else lf.loc[ts]))
            r["load_ramp_6h_target"] = float(lf.loc[ts] - (lf.loc[prev_6h_ts] if prev_6h_ts in lf.index else lf.loc[ts]))

            # day-level load stats
            r["load_day_mean"] = load_day_mean
            r["load_day_max"] = load_day_max
            r["load_day_min"] = load_day_min

            # targets/labels
            r["y_target"] = float(y.loc[ts])
            r["is_synthetic"] = int(day_flags.loc[ts, "is_synthetic"])
            r["is_observed"] = int(day_flags.loc[ts, "is_observed"])
            r["day"] = pd.Timestamp(day)
            r["ts"] = ts
            rows.append(r)

    ds = pd.DataFrame(rows).set_index("ts").sort_index()
    return ds

In [96]:
ds = build_daily_dataset(u)

# Feature columns
STATE_FEATURES = (
    ["last_y"]
    + [f"lag_{L}_t0" for L in STATE_LAGS]
    + ["ramp_1h_t0", "ramp_6h_t0", "ramp_1d_t0"]
    + [f"roll_mean_{w}_t0" for w in STATE_ROLL_WINS]
    + [f"roll_std_{w}_t0" for w in STATE_ROLL_WINS]
)

HORIZON_FEATURES = [
    "h", "q_in_hour_target", "qod_target", "hod_target", "dow_target", "month_target", "is_weekend_target",
    "load_fc_target", "load_ramp_1h_target", "load_ramp_6h_target",
    "load_day_mean", "load_day_max", "load_day_min",
]

FEATURE_COLS = STATE_FEATURES + HORIZON_FEATURES
ds = ds.dropna(subset=FEATURE_COLS + ["y_target"]).copy()

In [97]:
# ----------------------------
# 2) Define REAL days for evaluation
# ----------------------------
real_rows = ds[(ds["is_observed"] == 1) & (ds["is_synthetic"] == 0) & (ds.index >= BREAKPOINT)]
real_days = np.array(sorted(real_rows["day"].unique()))
if len(real_days) < TEST_DAYS + 5:
    raise ValueError(f"Not enough REAL days for TEST_DAYS={TEST_DAYS}. Have {len(real_days)} days.")

test_days = real_days[-TEST_DAYS:]
print(f"Phase 4 corrected: test days {test_days[0].date()} → {test_days[-1].date()} ({len(test_days)} days)")

Phase 4 corrected: test days 2026-01-22 → 2026-02-24 (30 days)


In [98]:
# ----------------------------
# 3) Train function: quantile models
# ----------------------------
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

def fit_quantile_models(train_df: pd.DataFrame, sample_weight: np.ndarray | None):
    models = {}
    for name, alpha in [("p10", 0.10), ("p50", 0.50), ("p90", 0.90)]:
        m = lgb.LGBMRegressor(objective="quantile", alpha=alpha, **LGB_PARAMS)
        m.fit(train_df[FEATURE_COLS], train_df["y_target"], sample_weight=sample_weight)
        models[name] = m
    return models

In [99]:
# ----------------------------
# 4) Rolling-origin evaluation (retrain as we move forward)
# ----------------------------
results = []
models1 = None
models2 = None

for i, D in enumerate(test_days):
    # Training data must be strictly BEFORE day D
    train_pool = ds[ds["day"] < D].copy()

    # Model 1: REAL only
    train1 = train_pool[(train_pool["is_observed"] == 1) & (train_pool["is_synthetic"] == 0)].copy()

    # Model 2: REAL + SYNTH with weights
    train2 = train_pool.copy()
    w2 = np.where(train2["is_synthetic"].values == 1, SYNTH_WEIGHT, 1.0).astype(float)

    # Retrain
    if (models1 is None) or (models2 is None) or ((i % RETRAIN_EVERY) == 0):
        models1 = fit_quantile_models(train1, sample_weight=None)
        models2 = fit_quantile_models(train2, sample_weight=w2)

    # Forecast rows for day D (96 rows)
    day_rows = ds[ds["day"] == D]
    X_day = day_rows[FEATURE_COLS]
    y_true = day_rows["y_target"].values
    

    p1_10 = models1["p10"].predict(X_day)
    p1_50 = models1["p50"].predict(X_day)
    p1_90 = models1["p90"].predict(X_day)

    p2_10 = models2["p10"].predict(X_day)
    p2_50 = models2["p50"].predict(X_day)
    p2_90 = models2["p90"].predict(X_day)

    idx = day_rows.index

    for ts, yt, a10, a50, a90, b10, b50, b90 in zip(idx, y_true, p1_10, p1_50, p1_90, p2_10, p2_50, p2_90):
        results.append({
            "day": D,
            "ts": ts,
            "y_true": float(yt),
            "m1_p10": float(a10), "m1_p50": float(a50), "m1_p90": float(a90),
            "m2_p10": float(b10), "m2_p50": float(b50), "m2_p90": float(b90),
        })

res = pd.DataFrame(results).set_index("ts").sort_index()

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002239 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3233
[LightGBM] [Info] Number of data points in the train set: 10752, number of used features: 30
[LightGBM] [Info] Start training from score 81.130997
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001084 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3233
[LightGBM] [Info] Number of data points in the train set: 10752, number of used features: 30
[LightGBM] [Info] Start training from score 110.910004
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002290 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3233
[LightGBM] [Info] Number of data points in the train set: 10752, number of used features: 30
[LightGBM] [Info] Start 

In [100]:
# ----------------------------
# 5) Metrics
# ----------------------------
def eval_point(y, yhat):
    mae = mean_absolute_error(y, yhat)
    rmse = float(np.sqrt(mean_squared_error(y, yhat)))
    return mae, rmse

In [101]:
m1_mae, m1_rmse = eval_point(res["y_true"], res["m1_p50"])
m2_mae, m2_rmse = eval_point(res["y_true"], res["m2_p50"])

m1_cov80 = float(((res["y_true"] >= res["m1_p10"]) & (res["y_true"] <= res["m1_p90"])).mean())
m2_cov80 = float(((res["y_true"] >= res["m2_p10"]) & (res["y_true"] <= res["m2_p90"])).mean())

thr = res["y_true"].quantile(0.99)
sp = res[res["y_true"] >= thr]
m1_sp_mae, m1_sp_rmse = eval_point(sp["y_true"], sp["m1_p50"])
m2_sp_mae, m2_sp_rmse = eval_point(sp["y_true"], sp["m2_p50"])

tmp = res.copy()
tmp["qod"] = (tmp.index.hour * 4 + (tmp.index.minute // 15)).astype(int)
m1_mae_by_qod = (tmp["y_true"] - tmp["m1_p50"]).abs().groupby(tmp["qod"]).mean()
m2_mae_by_qod = (tmp["y_true"] - tmp["m2_p50"]).abs().groupby(tmp["qod"]).mean()

print("\n============================")
print("PHASE 4 (corrected): Next-day 96-step curve forecast (one-shot), REAL evaluation only")
print("============================")
print(f"Test days: {test_days[0].date()} → {test_days[-1].date()} ({len(test_days)} days)")
print(f"SYNTH_WEIGHT (Model2): {SYNTH_WEIGHT}")

print("\nPoint forecast (P50):")
print(f"Model1 (REAL only)      MAE={m1_mae:.4f}  RMSE={m1_rmse:.4f}")
print(f"Model2 (REAL+SYNTH)     MAE={m2_mae:.4f}  RMSE={m2_rmse:.4f}")

print("\nUncertainty (P10-P90) coverage:")
print(f"Model1 coverage: {m1_cov80:.3f}")
print(f"Model2 coverage: {m2_cov80:.3f}")

print("\nSpike subset (top 1% of y_true in test):")
print(f"Threshold: {thr:.3f}  Count: {len(sp)}")
print(f"Model1 spike  MAE={m1_sp_mae:.4f}  RMSE={m1_sp_rmse:.4f}")
print(f"Model2 spike  MAE={m2_sp_mae:.4f}  RMSE={m2_sp_rmse:.4f}")

PHASE4_RESULTS = res
PHASE4_MAE_BY_QOD = pd.DataFrame({"m1": m1_mae_by_qod, "m2": m2_mae_by_qod})

print("\nPhase 4 corrected complete. Objects in memory:")
print(" - PHASE4_RESULTS")
print(" - PHASE4_MAE_BY_QOD")


PHASE 4 (corrected): Next-day 96-step curve forecast (one-shot), REAL evaluation only
Test days: 2026-01-22 → 2026-02-24 (30 days)
SYNTH_WEIGHT (Model2): 0.3

Point forecast (P50):
Model1 (REAL only)      MAE=17.6742  RMSE=30.5243
Model2 (REAL+SYNTH)     MAE=16.6889  RMSE=27.5997

Uncertainty (P10-P90) coverage:
Model1 coverage: 0.327
Model2 coverage: 0.474

Spike subset (top 1% of y_true in test):
Threshold: 375.834  Count: 29
Model1 spike  MAE=175.3194  RMSE=181.2134
Model2 spike  MAE=133.3406  RMSE=138.1636

Phase 4 corrected complete. Objects in memory:
 - PHASE4_RESULTS
 - PHASE4_MAE_BY_QOD
